# Figure 4 and Supplementary Figures 14-19: differential abundance

DESeq2 is **not** run here: the tables come from `code/01_compute/03_differential_abundance.ipynb`.

**Inputs** (input_dir)
- `metadata/sample_metadata.tsv`: 91 cfRNA samples (response, cohort)
- `metadata/multisource_metadata.tsv`: cfRNA + TME (Li et al. 2025) + PBMC (Maurer et al. 2024) samples, with DLBCL subtype
- `counts/counts_TMMCPM_cart_paper_ALLDATA.tsv.gz`: TMM-CPM of cfRNA and TME/PBMC pseudobulk samples (Fig4C, SFig16, SFig19)
- `counts/counts_norm_cart_paper.tsv.gz` + `reference/gencode.biotype.name.key.tsv.gz`: normalized cfRNA counts, Ensembl IDs mapped to symbols (Fig4B)
- `external/CTO_fractions_CART.tsv`: cfRNA cell-type deconvolution fractions (Tabula Sapiens reference)
- `external/Decruyenaere_supp_table4_DE_plasma.csv`: Decruyenaere et al. 2023, Supplementary Table 4 (plasma DE, R vs NR)

**Inputs** (results)
- `deseq/discovery_C1C2_all_genes.csv`, `deseq/cohort3_all_genes.csv`, `deseq/all_samples_all_genes.csv`, `deseq/subset_12mo_all_genes.csv`
- `ml/all/<method>/classifier_scores.csv`, `ml/12mo/<method>/classifier_scores.csv` (FMAC, TEX, LN, LASSO)

**Outputs** (plots/)
- `Fig4A_volcano.pdf`: volcano plot, discovery cohorts (C1+C2)
- `Fig4B_cfrna_boxplots.pdf`: LTB, OSBPL10, STAB1 in cfRNA by cohort
- `Fig4C_tme_pbmc_boxplots.pdf`: the same genes in TME and PBMC pseudobulk
- `SFig14_endothelial_fraction.pdf`, `SFig14_b_cell_fraction.pdf`: deconvolved cell fractions
- `SFig15_lfc_correlation_c3.pdf`: fold-change concordance, discovery vs Cohort 3
- `SFig16_cfrna_boxplots.pdf`: KLRB1, CDH5, DIPK2B in cfRNA by cohort
- `SFig17A_venn_all_vs_12mo.pdf`, `SFig17B_lfc_scatter_all_vs_12mo.pdf`, `SFig17C_auc_violin_all_vs_12mo.pdf`: sensitivity analysis, all samples vs ≥12-month follow-up
- `SFig18A_venn_vs_decruyenaere.pdf`, `SFig18B_padj_correlation_vs_decruyenaere.pdf`: comparison with Decruyenaere et al. 2023
- `SFig19_coo_markers_by_subtype.pdf`: cell-of-origin markers by DLBCL subtype in TME and cfRNA

In [ ]:
source("../../utils/setup.R")
suppressPackageStartupMessages({
    library(ggrepel)
    library(ggforce)
    library(ggvenn)
})

## Load inputs

In [ ]:
meta          <- load_sample_metadata()        # 91 cfRNA samples
ms_meta       <- load_multisource_metadata()   # cfRNA + TME + PBMC
counts_tmmcpm <- load_counts("tmmcpm_all")
counts_norm_sym <- ensembl_to_symbol(load_counts("norm"), load_gene_key())   # Fig4B

deconv       <- read.delim(in_path("external", "CTO_fractions_CART.tsv"))
# The header starts with a byte-order mark ("﻿Enseml_ID")
decruyenaere <- read.csv(in_path("external", "Decruyenaere_supp_table4_DE_plasma.csv"),
                         fileEncoding = "UTF-8-BOM")

read_deseq <- function(name) read.csv(res_in("deseq", paste0(name, "_all_genes.csv")))
deseq_disc <- read_deseq("discovery_C1C2")   # Cohorts 1+2
deseq_c3   <- read_deseq("cohort3")          # Cohort 3 (validation)
deseq_all  <- read_deseq("all_samples")      # all 91 samples
deseq_12mo <- read_deseq("subset_12mo")      # >=12-month follow-up (n = 85)

DAG_PADJ <- 0.1
dags <- deseq_disc %>% filter(!is.na(padj), padj < DAG_PADJ)
cat("Discovery DAGs (padj <", DAG_PADJ, "):", nrow(dags), "\n")
stopifnot(nrow(dags) == 776)

## Figure 4A — volcano plot, discovery cohorts

Points coloured when padj < 0.1 and |log2FC| > 0.5 (green = higher in responders).

In [ ]:
volcano_df <- deseq_disc %>%
    filter(!is.na(padj)) %>%
    mutate(sig_dir = case_when(padj < DAG_PADJ & log2FoldChange >  0.5 ~ "up",
                               padj < DAG_PADJ & log2FoldChange < -0.5 ~ "down",
                               TRUE                                    ~ "notsig"))
table(volcano_df$sig_dir)

genes_to_label <- c("OSBPL10", "STAB1", "LTB", "MS4A1", "IGHD")

volcano_plot <- ggplot(volcano_df, aes(x = log2FoldChange, y = -log10(padj), color = sig_dir)) +
    geom_point(size = 1, alpha = 0.75) +
    geom_hline(yintercept = -log10(DAG_PADJ), linewidth = 1, alpha = 0.5, linetype = "dashed") +
    geom_vline(xintercept = c(-0.5, 0.5), linewidth = 1, alpha = 0.5, linetype = "dashed") +
    scale_color_manual(values = c(notsig = "grey", up = RESPONSE_COLORS[["R"]],
                                  down = RESPONSE_COLORS[["NR"]])) +
    geom_text_repel(data = filter(volcano_df, gene_name %in% genes_to_label, padj > 0),
                    aes(x = log2FoldChange, y = -log10(padj), label = gene_name),
                    inherit.aes = FALSE, size = 2.5, box.padding = 0.4, point.padding = 0.3,
                    segment.color = "grey50", max.overlaps = Inf,
                    seed = 42) +
    theme_cfrna_print()
volcano_plot
save_plot(volcano_plot, "Fig4A_volcano", width = 2.25, height = 2.25)

## Figure 4B — LTB, OSBPL10 and STAB1 in cfRNA

The grey band marks the validation cohort. The same builder makes SFig16.
Fig4B plots the normalized counts (`counts_norm`, Ensembl IDs mapped to symbols) and SFig16 the TMM-CPM matrix of all samples (`counts_TMMCPM_ALLDATA`).
P-values: two-sided Wilcoxon rank-sum test, R vs NR within each cohort (normal approximation, `exact = FALSE`).

In [ ]:
# One gene's cfRNA TMM-CPM joined to the metadata. merge() sorts by sample_id, which fixes the
# row order and therefore the seeded jitter positions. response is made character, since
# position_jitterdodge() places the points differently for a factor.
cfrna_gene_df <- function(gene, counts) {
    merge(meta, t(counts[gene, meta$sample_id]), by.x = "sample_id", by.y = 0) %>%
        mutate(cohort   = sub("Cohort 3", "Cohort 3\n(validation)", cohort),
               response = as.character(response))
}

# stat_method = "wilcox.test" labels each cohort with its p-value, just above the highest point.
cfrna_gene_panel <- function(genes, counts, stat_method = NULL) {
    plots <- lapply(genes, function(g) {
        df <- cfrna_gene_df(g, counts)
        p <- response_boxplot(df, x = "cohort", y = g, title = g, y_lab = "TMM CPM",
                              jitter_width = 0.2, title_size = 8, axis_lines = TRUE,
                              shade_x = c(2.5, 3.5), stat_method = stat_method,
                              stat_label_y = max(df[[g]]) + 0.05 * diff(range(df[[g]])))
        if (!is.null(stat_method)) p <- p + scale_y_continuous(expand = expansion(mult = c(0.05, 0.08)))
        p
    })
    # only the left-most panel keeps its y-axis title
    plots[-1] <- lapply(plots[-1], `+`, theme(axis.title.y = element_blank()))
    Reduce(`+`, plots) + plot_layout(nrow = 1, guides = "collect")
}

fig4b <- cfrna_gene_panel(c("LTB", "OSBPL10", "STAB1"), counts_norm_sym, stat_method = "wilcox.test")

# The p-values and medians behind the panel
bind_rows(lapply(c("LTB", "OSBPL10", "STAB1"), function(g) {
    cfrna_gene_df(g, counts_norm_sym) %>%
        group_by(cohort) %>%
        summarise(gene = g, median_NR = median(.data[[g]][response == "NR"]),
                  median_R = median(.data[[g]][response == "R"]),
                  p = wilcox.test(.data[[g]][response == "R"], .data[[g]][response == "NR"], exact = FALSE)$p.value,
                  .groups = "drop")
}))
fig4b
save_plot(fig4b, "Fig4B_cfrna_boxplots", width = 4.8, height = 2.65)

## Figure 4C — LTB, OSBPL10 and STAB1 in TME and PBMC pseudobulk

TME: Li et al. 2025; PBMC: Maurer et al. 2024. TMM-CPM matrix of all samples. P-values: Wilcoxon rank-sum test, R vs NR
within each dataset (normal approximation).

In [ ]:
pseudobulk_gene_box <- function(gene) {
    df <- merge(ms_meta, t(counts_tmmcpm[gene, ms_meta$sample_id]), by.x = "sample_id", by.y = 0) %>%
        filter(dataset != "cfRNA") %>%
        mutate(dataset  = factor(dataset, levels = c("TME", "PBMC")),
               response = as.character(response),   # character (sets the jitter positions)
               cohort   = as.character(cohort))
    response_boxplot(df, x = "cohort", y = gene, y_lab = paste(gene, "TMM CPM"),
                     facet = "dataset", strip_size = 6, axis_lines = TRUE, stat_method = "wilcox.test") +
        scale_y_continuous(expand = expansion(mult = c(0.05, 0.12)))
}

for (g in c("LTB", "OSBPL10", "STAB1")) {
    df <- merge(ms_meta, t(counts_tmmcpm[g, ms_meta$sample_id]), by.x = "sample_id", by.y = 0) %>%
        filter(dataset != "cfRNA")
    print(compare_means(as.formula(paste(g, "~ response")), data = df, group.by = "dataset",
                        method = "wilcox.test", exact = FALSE))
}

fig4c <- Reduce(`+`, lapply(c("LTB", "OSBPL10", "STAB1"), pseudobulk_gene_box)) +
    plot_layout(nrow = 1, guides = "collect")
options(repr.plot.width = 6.5, repr.plot.height = 2)
fig4c
options(repr.plot.width = 6, repr.plot.height = 6)
save_plot(fig4c, "Fig4C_tme_pbmc_boxplots", width = 6.5, height = 2)

## Supplementary Figure 14 — deconvolved endothelial and B-cell fractions

Wilcoxon R vs NR per cell type across all 91 samples (BH-adjusted), then per-cohort box plots of the two cell types shown in the figure (per-cohort Wilcoxon p-values on the panels). All Wilcoxon tests use the normal approximation (`exact = FALSE`).

In [ ]:
deconv_long <- merge(meta[, c("sample_id", "response", "cohort")], deconv, by = "sample_id") %>%
    pivot_longer(-c(sample_id, response, cohort), names_to = "celltype", values_to = "fraction")

deconv_tests <- deconv_long %>%
    group_by(celltype) %>%
    summarise(pval = wilcox.test(fraction[response == "R"], fraction[response == "NR"], exact = FALSE)$p.value,
              .groups = "drop") %>%
    mutate(padj = p.adjust(pval, method = "BH")) %>%
    arrange(pval)
deconv_tests %>% filter(celltype %in% c("endothelial_cell", "b_cell"))
head(deconv_tests, 10)

In [ ]:
# The two panels differ only in the cell type and title.
deconv_boxplot <- function(celltype_name, title) {
    df <- deconv_long %>%
        filter(celltype == celltype_name) %>%
        mutate(group = factor(response, levels = c("R", "NR")))
    response_boxplot(df, x = "cohort", y = "fraction", group = "group", box_dodge = NULL,
                     title = title, y_lab = "Cell Fraction",
                     stat_method = "wilcox.test", stat_size = 2.5, stat_family = "Helvetica",
                     stat_label_y = 1.05 * max(df$fraction, na.rm = TRUE)) +
        theme_classic() %+replace%
        theme(plot.title   = element_text(size = 8, family = "Helvetica", color = "black"),
              axis.title.x = element_blank(),
              axis.title.y = element_text(size = 8, family = "Helvetica", angle = 90,
                                          margin = margin(r = 5), color = "black"),
              axis.text.x  = element_text(size = 8, family = "Helvetica", color = "black",
                                          margin = margin(t = 5)),
              axis.text.y  = element_text(size = 8, family = "Helvetica", color = "black",
                                          margin = margin(r = 5)),
              axis.line    = element_line(linewidth = 0.3),
              axis.ticks   = element_line(linewidth = 0.3),
              legend.position = "none") +
        scale_x_discrete(breaks = levels(df$cohort)) +
        scale_y_continuous(expand = expansion(mult = c(0.05, 0.1)))
}

sfig14_endo <- deconv_boxplot("endothelial_cell", "Endothelial Cell")
sfig14_b    <- deconv_boxplot("b_cell", "B Cell")
sfig14_endo + sfig14_b
# The p-values printed on the panels
deconv_long %>%
    filter(celltype %in% c("endothelial_cell", "b_cell")) %>%
    group_by(celltype, cohort) %>%
    summarise(p = wilcox.test(fraction[response == "R"], fraction[response == "NR"], exact = FALSE)$p.value,
              .groups = "drop")
save_plot(sfig14_endo, "SFig14_endothelial_fraction", width = 3, height = 3)
save_plot(sfig14_b, "SFig14_b_cell_fraction", width = 3, height = 3)

## Supplementary Figure 15 — fold-change concordance, discovery vs Cohort 3

log2FC of the discovery DAGs (padj < 0.1) in Cohorts 1+2 vs Cohort 3; genes not tested in Cohort 3 drop out of the join.
The subtitle states the padj < 0.1 filter (title and subtitle are blank in the saved panel).

In [ ]:
fc_c3 <- inner_join(deseq_disc %>% select(Row.names, log2FoldChange, padj),
                    deseq_c3 %>% select(Row.names, log2FoldChange, padj),
                    by = "Row.names", suffix = c(".disc", ".val")) %>%
    filter(!is.na(padj.disc), padj.disc < DAG_PADJ)
cat("Discovery DAGs tested in Cohort 3:", nrow(fc_c3), "\n")
cor.test(fc_c3$log2FoldChange.disc, fc_c3$log2FoldChange.val, method = "pearson")

sfig15 <- ggplot(fc_c3, aes(x = log2FoldChange.disc, y = log2FoldChange.val)) +
    geom_hline(yintercept = 0, linetype = "dashed", color = "grey60", linewidth = 0.4) +
    geom_vline(xintercept = 0, linetype = "dashed", color = "grey60", linewidth = 0.4) +
    geom_point(size = 1, alpha = 0.6, color = "#1F3B73") +
    geom_smooth(method = "lm", formula = y ~ x, color = "#B22222", fill = "#B22222",
                alpha = 0.15, linewidth = 0.5, se = TRUE) +
    stat_cor(method = "pearson", label.x = -0.9 * 3, label.y = 0.9 * 2.5, size = 2) +
    coord_fixed(xlim = c(-3, 5), ylim = c(-2.5, 2.5)) +
    theme_cfrna_print() +
    theme(plot.title = element_blank(), plot.subtitle = element_blank(),
          axis.text = element_text(color = "black"),
          panel.border = element_rect(color = "black", fill = NA, linewidth = 0.8)) +
    labs(title = "Concordance of Effect Sizes Between Discovery and Validation Cohorts",
         subtitle = paste0("Discovery FDR < ", DAG_PADJ, " (n = ", nrow(fc_c3), " genes)"),
         x = "Log2 Fold Change (Discovery: Cohorts 1+2)",
         y = "Log2 Fold Change (Validation: Cohort 3)")
sfig15
save_plot(sfig15, "SFig15_lfc_correlation_c3", width = 3, height = 2.5)

## Supplementary Figure 16 — KLRB1, CDH5 and DIPK2B in cfRNA

In [ ]:
sfig16 <- cfrna_gene_panel(c("KLRB1", "CDH5", "DIPK2B"), counts_tmmcpm)
sfig16
save_plot(sfig16, "SFig16_cfrna_boxplots", width = 4.8, height = 2.65)

## Supplementary Figure 17 — sensitivity analysis: all samples vs ≥12-month follow-up

DESeq2 R vs NR on all 91 samples and on the 85 samples with ≥12 months of follow-up (padj < 0.1 in each).

In [ ]:
stopifnot(nrow(filter_12mo(meta)) == 85)

sens_cmp <- inner_join(
    deseq_all  %>% select(Row.names, gene_name, lfc_all = log2FoldChange, padj_all = padj),
    deseq_12mo %>% select(Row.names, lfc_12mo = log2FoldChange, padj_12mo = padj),
    by = "Row.names") %>%
    mutate(sig_all   = !is.na(padj_all)  & padj_all  < DAG_PADJ,
           sig_12mo  = !is.na(padj_12mo) & padj_12mo < DAG_PADJ,
           sig_class = case_when(sig_all & sig_12mo  ~ "Both",
                                 sig_all & !sig_12mo ~ "All Samples",
                                 !sig_all & sig_12mo ~ "12mo only",
                                 TRUE                ~ "Neither"))

n_all  <- sum(sens_cmp$sig_all)
n_12mo <- sum(sens_cmp$sig_12mo)
n_both <- sum(sens_cmp$sig_all & sens_cmp$sig_12mo)
c(all = n_all, mo12 = n_12mo, both = n_both, all_only = n_all - n_both, mo12_only = n_12mo - n_both)

### SFig17A — Venn diagram

Area-proportional two-circle Euler diagram: circle areas scale with the DAG counts and the centre distance is solved so that the lens area matches the overlap.

In [ ]:
# Area of the intersection of two circles with radii r1, r2 whose centres are d apart.
lens_area <- function(d, r1, r2) {
    if (d <= abs(r1 - r2)) return(pi * min(r1, r2)^2)
    if (d >= r1 + r2) return(0)
    a1 <- acos(pmax(-1, pmin(1, (d^2 + r1^2 - r2^2) / (2 * d * r1))))
    a2 <- acos(pmax(-1, pmin(1, (d^2 + r2^2 - r1^2) / (2 * d * r2))))
    r1^2 * a1 + r2^2 * a2 -
        0.5 * sqrt(max(0, (-d + r1 + r2) * (d + r1 - r2) * (d - r1 + r2) * (d + r1 + r2)))
}

r1 <- 1
r2 <- sqrt(n_12mo / n_all)
d  <- optimize(function(x) (lens_area(x, r1, r2) - (n_both / n_all) * pi * r1^2)^2,
               interval = c(1e-4, r1 + r2 - 1e-4))$minimum

x_left_ctr  <- (-r1 + (d - r2)) / 2   # centre of the "all samples only" crescent
x_right_ctr <- (r1 + (d + r2)) / 2    # centre of the "12mo only" sliver
x_overlap   <- ((d - r2) + r1) / 2    # centre of the overlap lens
x_lbl_l <- -r1 - 0.32                 # label anchors outside the circles
x_lbl_r <-  d + r2 + 0.32

venn_label <- function(x, label, n, hjust) {
    list(annotate("text", x = x, y =  0.08, label = label, size = 4, hjust = hjust, color = "grey20"),
         annotate("text", x = x, y = -0.09, label = n,     size = 4, hjust = hjust, color = "grey20"))
}

sfig17a <- ggplot() +
    geom_circle(aes(x0 = 0, y0 = 0, r = r1), fill = "#2c7bb6", alpha = 0.75, color = NA) +
    geom_circle(aes(x0 = d, y0 = 0, r = r2), fill = "grey", alpha = 0.75, color = NA) +
    venn_label(x_overlap, "Overlap", n_both, 0.5) +
    annotate("segment", x = x_left_ctr, xend = x_lbl_l + 0.05, y = 0, yend = 0,
             color = "grey50", linewidth = 0.3) +
    venn_label(x_lbl_l, "All Samples", n_all - n_both, 1) +
    annotate("segment", x = x_right_ctr, xend = x_lbl_r - 0.05, y = 0, yend = 0,
             color = "grey50", linewidth = 0.3) +
    venn_label(x_lbl_r, ">12mo only", n_12mo - n_both, 0) +
    coord_equal(clip = "off") +
    theme_void()
sfig17a
save_plot(sfig17a, "SFig17A_venn_all_vs_12mo", width = 7, height = 5)

### SFig17B — log2FC concordance

In [ ]:
r_sens <- cor(sens_cmp$lfc_all, sens_cmp$lfc_12mo, use = "complete.obs")
cat("Pearson r (all genes):", round(r_sens, 4), "\n")

sfig17b <- ggplot(sens_cmp, aes(lfc_all, lfc_12mo, color = sig_class)) +
    geom_point(size = 1.5, alpha = 0.5) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey40") +
    scale_color_manual(values = c("Both" = "#2c7bb6", "All Samples" = "#d7191c",
                                  "12mo only" = "#1a9641", "Neither" = "grey80")) +
    labs(title = sprintf("LFC concordance (r = %.3f)", r_sens), color = "Significant in") +
    theme_classic(base_size = 12)
sfig17b
save_plot(sfig17b, "SFig17B_lfc_scatter_all_vs_12mo", width = 4, height = 3.5)

### SFig17C — test AUC, all patients vs ≥12-month cohort

Monte Carlo CV test AUCs (101 iterations) from `results/ml/all` and `results/ml/12mo`; Wilcoxon test per signature (normal approximation).
The title uses "≥", so this panel is written with `cairo_pdf` (the default pdf device printed "...").

In [ ]:
METHODS_SENS <- c("FMAC", "TEX", "LN", "LASSO")
SENS_LABELS  <- c("All patients", ">12mo follow-up")
sens_colors  <- setNames(SENSITIVITY_COLORS, SENS_LABELS)
auc_meta     <- meta[, c("sample_id", "response")]

auc_sens <- bind_rows(
    roc_by_iteration(res_in("ml", "all"),  METHODS_SENS, auc_meta) %>% mutate(cohort = SENS_LABELS[1]),
    roc_by_iteration(res_in("ml", "12mo"), METHODS_SENS, auc_meta) %>% mutate(cohort = SENS_LABELS[2])
) %>%
    mutate(method = recode(method, LASSO = "All Transcripts"),
           method = factor(method, levels = c("TEX", "FMAC", "LN", "All Transcripts")),
           cohort = factor(cohort, levels = SENS_LABELS),
           test   = test / 100)

auc_sens %>%
    group_by(method, cohort) %>%
    summarise(median_test_auc = median(test), n = n(), .groups = "drop")
auc_sens %>%
    group_by(method) %>%
    summarise(wilcox_p = wilcox.test(test[cohort == SENS_LABELS[1]], test[cohort == SENS_LABELS[2]],
                                     exact = FALSE)$p.value)

sfig17c <- ggplot(auc_sens, aes(x = cohort, y = test, fill = cohort, color = cohort)) +
    geom_violin(width = 0.8, alpha = 0.7, linewidth = 0.4) +
    stat_summary(fun = median, geom = "point", size = 2.5,
                 fill = "black", color = "black", stroke = NA, alpha = 1.0) +
    stat_compare_means(comparisons = list(SENS_LABELS), method = "wilcox.test", method.args = WILCOX_ARGS,
                       label = "p.signif", hide.ns = FALSE, size = 2.5, tip.length = 0) +
    facet_wrap(~method, nrow = 1) +
    scale_fill_manual(values = sens_colors) +
    scale_color_manual(values = sens_colors) +
    labs(x = NULL, y = "Test AUC", title = "Sensitivity analysis: ≥12mo follow-up") +
    theme_classic(base_size = 9) +
    theme(legend.position  = "bottom",
          legend.text      = element_text(size = 10),
          legend.title     = element_blank(),
          strip.background = element_blank(),
          strip.text       = element_text(size = 10),
          axis.text.y      = element_text(size = 10),
          axis.text.x      = element_blank(),
          axis.ticks.x     = element_blank())
sfig17c
save_plot(sfig17c, "SFig17C_auc_violin_all_vs_12mo", width = 5, height = 3, device = cairo_pdf)

## Supplementary Figure 18 — comparison with Decruyenaere et al. 2023

Our 776 discovery DAGs vs the 504 R-vs-NR DAGs in plasma cfRNA of first-line chemoimmunotherapy-treated DLBCL, matched on unversioned Ensembl ID (padj < 0.1 in both).
The DAG list is taken from `discovery_C1C2_all_genes.csv` (776 genes).

In [ ]:
loy <- dags %>%
    mutate(ensembl_id = sub("\\..*", "", Row.names)) %>%
    select(ensembl_id, gene_name, log2FoldChange, pvalue, padj)
decru <- decruyenaere %>%
    rename(ensembl_id = Enseml_ID) %>%
    select(ensembl_id, gene_name, p, padj)

loy_sig   <- loy   %>% filter(!is.na(padj), padj < DAG_PADJ) %>% pull(ensembl_id)
decru_sig <- decru %>% filter(!is.na(padj), padj < DAG_PADJ) %>% pull(ensembl_id)
shared    <- intersect(loy_sig, decru_sig)
c(loy = length(loy_sig), decruyenaere = length(decru_sig), shared = length(shared))

In [ ]:
sfig18a <- ggvenn(list("Loy et al" = loy_sig, "Decruyenaere et al, 2023" = decru_sig),
                  fill_color = unname(RESPONSE_COLORS[c("R", "NR")]),
                  stroke_size = 0.5, set_name_size = 4, text_size = 4)
sfig18a
save_plot(sfig18a, "SFig18A_venn_vs_decruyenaere", width = 4, height = 4)

In [ ]:
shared_df <- loy %>%
    filter(ensembl_id %in% shared) %>%
    select(ensembl_id, gene_name, padj_loy = padj) %>%
    inner_join(decru %>% select(ensembl_id, padj_decru = padj), by = "ensembl_id") %>%
    mutate(neg_log10_padj_loy = -log10(padj_loy), neg_log10_padj_decru = -log10(padj_decru))
shared_df

# Spearman: the adjusted p-values are skewed and extreme-valued (ties -> no exact p-value)
suppressWarnings(cor.test(shared_df$neg_log10_padj_loy, shared_df$neg_log10_padj_decru,
                          method = "spearman"))

sfig18b <- ggplot(shared_df, aes(x = neg_log10_padj_loy, y = neg_log10_padj_decru)) +
    geom_point(size = 2, alpha = 1, color = FOREST_BLUE) +
    geom_smooth(method = "lm", formula = y ~ x, se = TRUE, color = "grey30", linewidth = 0.5) +
    labs(x = "-log10(padj): Loy et al", y = "-log10(padj): Decruyenaere et al") +
    theme_bw()
sfig18b
save_plot(sfig18b, "SFig18B_padj_correlation_vs_decruyenaere", width = 3, height = 3)

## Supplementary Figure 19 — cell-of-origin markers by DLBCL subtype in TME and cfRNA

BCL2, BCL6, MME (CD10) and MYC in GCB vs non-GCB samples (TME: 8 / 7, cfRNA: 42 / 32); samples with subtype "Unknown" or no subtype are excluded, as in Table S1.
P-values: Wilcoxon rank-sum test, GCB vs non-GCB within each dataset (normal approximation).
`geom_jitter()` is unseeded, so the seed is set before saving.

In [ ]:
COO_GENES      <- c("BCL2", "BCL6", "MME", "MYC")
SUBTYPE_COLORS <- c(GCB = RESPONSE_COLORS[["R"]], "Non-GCB" = RESPONSE_COLORS[["NR"]])
COO_TITLES     <- list(BCL2 = quote(italic("BCL2")), BCL6 = quote(italic("BCL6")),
                       MME = quote(italic("MME") ~ "(CD10)"), MYC = quote(italic("MYC")))

coo_gene_df <- function(gene) {
    merge(ms_meta, t(counts_tmmcpm[gene, ms_meta$sample_id]), by.x = "sample_id", by.y = 0) %>%
        filter(dataset != "PBMC", subtype %in% c("GCB", "Non-GCB")) %>%
        mutate(dataset = factor(dataset, levels = c("TME", "cfRNA")),
               subtype = factor(subtype, levels = c("GCB", "Non-GCB")))
}

# One gene per panel pair (TME, cfRNA): p-value on top, group sizes at the bottom
coo_gene_box <- function(gene) {
    df <- coo_gene_df(gene)
    response_boxplot(df, x = "subtype", y = gene, group = "subtype", palette = SUBTYPE_COLORS,
                     y_lab = "TMM CPM", title = as.expression(COO_TITLES[[gene]]),
                     box_linewidth = 0.25, jitter_dodge = FALSE, jitter_width = 0.2,
                     facet = "dataset", strip_size = 6, strip_margin = TRUE,
                     stat_method = "wilcox.test", stat_size = 2.5, stat_label_x_npc = "center") +
        geom_text(data = count(df, dataset, subtype), aes(x = subtype, y = -Inf, label = paste0("n=", n)),
                  inherit.aes = FALSE, vjust = -0.4, size = 2) +
        theme(panel.grid.major.y = element_line(color = "grey92", linewidth = 0.4),
              axis.title.y = element_text(size = 10, angle = 90),
              plot.title = element_text(hjust = 0.5, size = 12))
}

table(coo_gene_df("BCL2")$dataset, coo_gene_df("BCL2")$subtype)
bind_rows(lapply(COO_GENES, function(g) {
    compare_means(as.formula(paste(g, "~ subtype")), data = coo_gene_df(g), group.by = "dataset",
                  method = "wilcox.test", exact = FALSE)
}))

sfig19 <- Reduce(`+`, lapply(COO_GENES, coo_gene_box)) + plot_layout(nrow = 2)
set.seed(42)
save_plot(sfig19, "SFig19_coo_markers_by_subtype", width = 5.3, height = 5.3)
sfig19